# PINTP network traffic prediction

Run the graph ADR benchmark and the sparse-label telemetry benchmark from the same implementation used by the CLI. Place this notebook beside the `pintp/` folder and `requirements.txt`. In Colab, upload and extract the project ZIP, then use `%cd /content/TrafficAI` if that is where it was extracted.

The default quick profile runs all experiments for one seed. Set `QUICK = False` for 100 graph epochs, 2,000 telemetry epochs and three seeds. The telemetry source is synthetic unless a real `data/machine_usage.csv` file is supplied. See `APPROACH.md` for the equations and assumptions.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import tempfile

assert Path('pintp').is_dir(), 'Change the notebook working directory to the project root.'
if 'google.colab' in sys.modules:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'pintp-matplotlib'))
from IPython.display import Image, display
import pandas as pd
from pintp.experiment import Config, run

QUICK = True
TRACE = Path('data/machine_usage.csv')
MACHINE = 'm_2274'
DEVICE = 'cpu'  # Change to 'cuda' if a CUDA runtime is available.
Path('runs').mkdir(exist_ok=True)
OUTPUT = Path(tempfile.mkdtemp(prefix='pintp-', dir='runs'))
config = Config(output=OUTPUT, csv=TRACE if TRACE.is_file() else None,
                machine=MACHINE, profile='quick' if QUICK else 'full', device=DEVICE)

## Train and evaluate

The graph benchmark uses complete 100-node time slices. The telemetry benchmark uses 287 labeled rows and training-only collocation. Both reserve the final chronological window for testing.

In [ ]:
info = run(config)
display(pd.Series(info, name='Run configuration'))

## Benchmark metrics and ablations

In [ ]:
source = info['source']
for name in ('synthetic_results', 'synthetic_lambda_sweep', 'synthetic_robustness',
             f'{source}_results', f'{source}_ablation', f'{source}_coefficients'):
    print(name)
    display(pd.read_csv(OUTPUT / f'{name}_summary.csv'))

## Forecast plots

In [ ]:
for name in ('graph_forecast', 'telemetry_forecast', 'telemetry_ablation'):
    display(Image(filename=str(OUTPUT / f'{name}.png')))

## Reload the trained graph forecaster

In [ ]:
from pintp.predict import predict_graph

display(predict_graph(OUTPUT / 'graph_pintp.pt', [8.0, 9.0, 10.0]).head())
print(f'CSV files, figures, model checkpoints and metadata: {OUTPUT}')